In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.io import load_dataset, load_fine


DATA_DIR = REPO_ROOT / "DATA"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

fine = load_fine(
    DATA_DIR,
    "fine.csv",
)

print(f"Repository root : {REPO_ROOT}")
print(f"Data directory  : {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Load the aggregated regional data and 

aggregated_mid = load_dataset(
    data_dir = DATA_DIR,
    filename = "aggregated_mid.csv",
    index_col=0
)

aggregation_dictionary = pd.read_csv(
    DATA_DIR / "dictionary_for_aggregated.csv",
    sep=";"
)


allen_coarse_colors = {
    "Isocortex": "#70ff71",
    "OLF": "#9ad2bd",
    "HPF": "#7ed04b",
    "CTXsp": "#8ada87",
    "STR": "#98d6f9",
    "PAL": "#8599cc",
    "TH": "#ff7080",
    "HY": "#e64438",
    "MB": "#ff64ff",
    "P": "#ff9b88",
    "MY": "#ff9bcd"
}


conditions = ['CNTX','OCT',"OPCRT"]

condition_datasets = {
    condition: aggregated_mid.filter(
        regex=rf"{condition}",
        axis=1
    )
    for condition in conditions
}

In [ ]:
# Compute correlation matrices for each condition and visualize their
# structure using heatmaps and a two-dimensional UMAP embedding

import umap

plt.rcParams["font.family"] = "Arial"

palette = {
    "CNTX": "#66c2a5",
    "OCT": "#fc8d62",
    "OPCRT": "#8da0cb"
}

corr_matrices = {}

for condition_name, condition_data in condition_datasets.items():

    correlation_matrix = condition_data.T.corr()
    corr_matrices[condition_name] = correlation_matrix

    fig, ax = plt.subplots(figsize=(12, 9))

    heatmap = sns.heatmap(
        correlation_matrix,
        cmap="crest",
        vmin=-1,
        vmax=1,
        xticklabels=True,
        yticklabels=True,
        ax=ax
    )

    ax.tick_params(axis="x", rotation=90, labelsize=10)
    ax.tick_params(axis="y", labelsize=10)

    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)

    ax.tick_params(
        left=False,
        bottom=False
    )

    colorbar = heatmap.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=17)
    colorbar.outline.set_linewidth(2)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"{condition_name}_correlation_heatmap.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()



X_all = []
labels = []
regions = []

for name, C in corr_matrices.items():

    if name == "HC":
        continue

    X_all.append(C.fillna(0).values)
    labels.extend([name] * C.shape[0])
    regions.extend(C.index)

X_all = np.vstack(X_all)

reducer = umap.UMAP(
    n_components=2,
    metric="correlation",
    n_neighbors=15,
    min_dist=0.05,
    random_state=42
)

emb_all = reducer.fit_transform(X_all)

emb_df = pd.DataFrame(
    emb_all,
    columns=["UMAP1", "UMAP2"]
)

emb_df["condition"] = labels
emb_df["region"] = regions



plt.figure(figsize=(9, 7))

for condition, color in palette.items():

    subset = emb_df[emb_df["condition"] == condition]

    plt.scatter(
        subset["UMAP1"],
        subset["UMAP2"],
        label=condition,
        color=color,
        s=80,
        alpha=0.8
    )

plt.xlabel("UMAP1", fontsize=14)
plt.ylabel("UMAP2", fontsize=14)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.legend(
    fontsize=12,
    frameon=False
)

plt.grid(False)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "correlation_matrices_UMAP.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Compute adjacency matrices for each experimental condition

from utils.network import compute_adj_matrix

adj_matrices = {}

for name, df in condition_datasets.items():
    adj_matrices[name] = compute_adj_matrix(df)

In [ ]:
# Build functional networks

from utils.network import build_brain_graph

brain_graphs = {}

for name, matrix in adj_matrices.items():
    results = build_brain_graph(
        adj_matrix=matrix,
        aggregated_codex=aggregation_dictionary,
        allen_coarse_colors=allen_coarse_colors,
        min_dist=0.5,
        iterations=60,
        seed=42,
        super_scale=5,
        group_scale=1,
        node_size=200,
        SAVE=False
    )
    brain_graphs[name] = results

In [ ]:
# Bootstrap graph metrics across subjects for each experimental condition

"""
from utils.network import build_brain_graph_no_plot, compute_graph_metrics, bootstrap_adj_matrix, bootstrap_graph_metrics_subjects

dfs_sub = {k: v for k, v in condition_datasets.items() if k in ['CNTX','OCT','OPCRT']}
df_metrics_dist_all = {}

for name, df in dfs_sub.items():
    print(f"Subject bootstrap for graph: {name}")
    df_metrics = bootstrap_graph_metrics_subjects(df, aggregation_dictionary, allen_coarse_colors)
    df_metrics_dist_all[name] = df_metrics
"""

In [ ]:
# Load bootstrapped network metrics

FIGURE_DIR = REPO_ROOT / "Fig_4"

cntx_metrics = pd.read_csv(
    FIGURE_DIR / "cntx_bootstrapped_metrics.csv",
    sep=";",
    index_col=0,
)

oct_metrics = pd.read_csv(
    FIGURE_DIR / "oct_bootstrapped_metrics.csv",
    sep=";",
    index_col=0,
)

opcrt_metrics = pd.read_csv(
    FIGURE_DIR / "opcrt_bootstrapped_metrics.csv",
    sep=";",
    index_col=0,
)

In [ ]:
df_plot = pd.concat(
    [
        cntx_metrics.assign(network='CNTX'),
        oct_metrics.assign(network='OCT'),
        opcrt_metrics.assign(network='OPCRT')
    ],
    ignore_index=True
)

df_melt = df_plot.melt(
    id_vars='network',
    var_name='metric',
    value_name='value'
)

metrics = df_melt['metric'].unique()

In [ ]:
# Plot the bootstrap distributions of network metrics

palette = {
    'CNTX': '#66c2a5',
    'OCT': '#fc8d62',
    'OPCRT': '#8da0cb'
}

plt.rcParams['font.family'] = 'Arial'

names = {
    'density': 'Density',
    'degree': 'Degree',
    'global_efficiency': 'Global efficiency',
    'clustering_coefficient': 'Clustering coefficient',
    'average_betweenness_centrality': 'Betweenness centrality',
    'sigma (small_worldness)': 'Small-worldness'
}

for m in metrics:
    plt.figure(figsize=(5, 4))

    df_sub = df_melt[df_melt['metric'] == m]

    max_val = df_sub['value'].max()
    y_max = max_val * 1.3

    sns.violinplot(
        data=df_sub,
        x='network',
        y='value',
        hue='network',
        palette=palette,
        legend=False
    )

    plt.ylabel(names[m], fontsize=14)
    plt.xlabel('')
    plt.xticks(rotation=0, fontsize=14)
    plt.yticks(fontsize=12)
    plt.ylim(0, y_max)

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"{m}_violinplot.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# Compare network metrics across experimental conditions 
# using Kruskal–Wallis tests and Dunn's post-hoc tests

from utils.statistics import kruskal_wallis_dunn

results_kw, results_dunn = kruskal_wallis_dunn(
    df_melt,
    metric_col="metric",
    group_col="network",
    value_col="value",
    groups=("CNTX", "OCT", "OPCRT"),
    p_adjust="bonferroni"
)

results_dunn

In [ ]:
# Classify OPCRT network regions according to
# their within-module connectivity and participation across modules

import bct  # bctpy
from scipy.stats import zscore
import community as community_louvain
import networkx as nx

G = brain_graphs['OPCRT']['G'] #using OPCRT group
A = nx.to_numpy_array(G)  

partition = community_louvain.best_partition(G,random_state=42)     
nodes = list(G.nodes())
membership = np.array([partition[n] for n in nodes])

within_z = bct.module_degree_zscore(A, membership)
pc = bct.participation_coef(A, membership, degree='undirected')

roles = []
for i in range(len(nodes)):
    z = within_z[i]
    p = pc[i]
    if z < 1:  # non-hub
        if p <= 0.05:
            roles.append("R1: Ultra-peripheral")
        elif p <= 0.62:
            roles.append("R2: Peripheral")
        elif p <= 0.80:
            roles.append("R3: Non-hub connector")
        else:
            roles.append("R4: Non-hub kinless")
    else:  # hub
        if p <= 0.30:
            roles.append("R5: Provincial hub")
        elif p <= 0.75:
            roles.append("R6: Connector hub")
        else:
            roles.append("R7: Kinless hub")


df_roles = pd.DataFrame({
    "Node": nodes,
    "WithinModule_Z": within_z,
    "Participation_Coef": pc,
    "Role": roles
})

In [ ]:
# Load the network hub classifications

df_roles_cntx = pd.read_csv(
    FIGURE_DIR / "hub_cntx.csv",
    sep=";",
    index_col=0,
)

df_roles_oct = pd.read_csv(
    FIGURE_DIR / "hub_oct.csv",
    sep=";",
    index_col=0,
)

df_roles_opcrt = pd.read_csv(
    FIGURE_DIR / "hub_opcrt.csv",
    sep=";",
    index_col=0,
)

In [ ]:
# Visualize the participation coefficient and within-module degree of network nodes,
# classified by their topological roles

from adjustText import adjust_text
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Arial"

conditions = {
    "CNTX": df_roles_cntx,
    "OCT": df_roles_oct,
    "OPCRT": df_roles_opcrt
}

role_list = [
    "R1: Ultra-peripheral",
    "R2: Peripheral",
    "R3: Non-hub connector",
    "R4: Non-hub kinless",
    "R5: Provincial hub",
    "R6: Connector hub",
    "R7: Kinless hub"
]

dot_colors = [
    "#bbd6e8",
    "#c1e2bf",
    "#f6b9ba",
    "#fff2b2",
    "#d2c4e0",
    "#ffd8b2",
    "#e4ccbe"
]

label_colors = [
    "#1F78B4",
    "#33A02C",
    "#E31A1C",
    "#FFD700",
    "#6A3D9A",
    "#FF7F00",
    "#A65628"
]

for condition_name, df_roles in conditions.items():

    plt.figure(figsize=(12, 6))

    all_texts = []

    for role, dot_color, label_color in zip(
        role_list, dot_colors, label_colors
    ):

        idx = df_roles["Role"] == role

        x = df_roles.loc[idx, "Participation_Coef"].values
        y = df_roles.loc[idx, "WithinModule_Z"].values
        nodes_in_role = df_roles.loc[idx, "Node"].values

        plt.scatter(
            x,
            y,
            label=role,
            s=400,
            color=dot_color
        )

        for xi, yi, node in zip(x, y, nodes_in_role):

            t = plt.text(
                xi,
                yi,
                str(node),
                fontsize=10,
                fontweight="bold",
                color=label_color
            )

            all_texts.append(t)

    adjust_text(
        all_texts,
        only_move={"points": "y", "texts": "xy"},
        autoalign="xy",
        expand_points=(1.2, 1.4),
        expand_text=(1.2, 1.4),
        force_points=0.05,
        force_text=0.05,
        arrowprops=dict(
            arrowstyle="-",
            color="none"
        )
    )

    plt.xlabel(
        "Participation Coefficient (P)",
        fontsize=14
    )

    plt.ylabel(
        "Within-Module Degree Z-score (z)",
        fontsize=14
    )

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    plt.legend(
        bbox_to_anchor=(1.01, 1),
        loc="upper left",
        fontsize=12,
        labelspacing=1.2
    )

    plt.grid(False)

    plt.title(
        condition_name,
        fontsize=16,
        fontweight="bold"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"hub_roles_{condition_name.lower()}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()